# DynGraphEval

Evaluates temporal graph models on two dimensions:
1. **Standard MRR** — TGB `hist_rnd` negatives (leaderboard-comparable)
2. **Recency MRR** — negatives from the source node's recent interaction history

**To switch models:** edit `config.yaml`, then re-run all cells.

## 0. Setup
Mounts Google Drive (for checkpoints/datasets) and clones the latest code from GitHub.

Works in both the Colab browser and the **VS Code Colab extension**.

**Before running:** make sure `DynGraphEval/` is in your Google Drive at `MyDrive/DynGraphEval`.

In [ ]:
import sys, os, subprocess

# ── 1. Mount Drive (persistent storage: checkpoints, datasets, neg_cache) ──
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/DynGraphEval"
CODE_DIR  = "/content/DynGraphEval"
REPO_URL  = "https://github.com/importjose/DynGraphEval.git"
BRANCH    = "feature/init"

# ── 2. Clone or pull latest code from GitHub ───────────────────────────────
if os.path.exists(f"{CODE_DIR}/.git"):
    subprocess.run(f"git -C {CODE_DIR} pull --quiet", shell=True, check=True)
else:
    subprocess.run(
        f"git clone --quiet --branch {BRANCH} {REPO_URL} {CODE_DIR}",
        shell=True, check=True
    )

os.chdir(CODE_DIR)
print(f"Working directory: {os.getcwd()}")

# ── 3. Symlink persistent dirs from Drive into the repo ────────────────────
# Keeps large files (datasets, caches, checkpoints) on Drive across restarts
for rel in ["datasets", "neg_cache", "models/tgn/checkpoints"]:
    src = os.path.join(DRIVE_DIR, rel)
    dst = os.path.join(CODE_DIR, rel)
    os.makedirs(src, exist_ok=True)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        os.symlink(src, dst)

# ── 4. Install dependencies ────────────────────────────────────────────────
subprocess.run("pip install -q torch-geometric py-tgb", shell=True, check=True)

# ── 5. Make dyngrapheval importable ───────────────────────────────────────
src_path = os.path.join(CODE_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import dyngrapheval, torch
print(f"dyngrapheval ready  |  {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 1. Imports

In [ ]:
import json
import yaml
import torch
import numpy as np

from tgb.linkproppred.dataset_pyg import PyGLinkPropPredDataset

from dyngrapheval import Evaluator
from dyngrapheval.models import TGN, FederatedTGN, FedLink, TPNetTGN

## 2. Load Config

In [ ]:
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

# Read all settings (with defaults for optional fields)
model_type    = cfg["model"]                        # tgn | fl_tgn | fedlink
checkpoints   = cfg["checkpoints"]                  # str or list[str]
model_name    = cfg.get("name",      model_type)
dataset_name  = cfg.get("dataset",   "tgbl-wiki")
seed          = cfg.get("seed",      42)
neg_cache_dir = cfg.get("cache_dir", "neg_cache")
device_str    = cfg.get("device",    None)          # None = auto-detect

# Normalize checkpoints to always be a list
if isinstance(checkpoints, str):
    checkpoints = [checkpoints]

print(f"Model:    {model_name} ({model_type})")
print(f"Dataset:  {dataset_name}")
print(f"Checkpoints: {checkpoints}")

## 3. Device & Dataset

In [ ]:
# ── Device ────────────────────────────────────────────────────────────────────
if device_str:
    device = torch.device(device_str)
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

# ── Dataset ───────────────────────────────────────────────────────────────────
dataset = PyGLinkPropPredDataset(name=dataset_name, root="datasets")
data    = dataset.get_TemporalData()

# Temporal splits — masks live on dataset, used to index into data
train_data = data[dataset.train_mask]
val_data   = data[dataset.val_mask]
test_data  = data[dataset.test_mask]

# Metadata
num_nodes   = dataset.num_nodes
msg_dim     = data.msg.shape[-1]          # edge feature dimension from data directly
min_dst_idx = int(data.dst.min().item())
max_dst_idx = int(data.dst.max().item())
num_users   = int(data.src.max().item()) + 1
num_pages   = max_dst_idx - min_dst_idx + 1

print(f"Nodes: {num_nodes}  |  Edges: {data.num_events}")
print(f"Train: {train_data.num_events}  Val: {val_data.num_events}  Test: {test_data.num_events}")
print(f"msg_dim: {msg_dim}  |  dst range: [{min_dst_idx}, {max_dst_idx}]")

## 4. Instantiate Model

In [ ]:
if model_type == "tgn":
    # TGN (TPNet implementation) — 2-layer GraphAttention, full historical NeighborSampler.
    # Checkpoint format: nn.Sequential(backbone, link_pred).state_dict() .pkl
    # Trained using lxd99/TGB_TPNet train_link_prediction.py with model_name=TGN.
    model = TPNetTGN(
        checkpoint_path = checkpoints[0],
        num_nodes       = num_nodes,
        msg_dim         = msg_dim,
        train_data      = train_data,
        val_data        = val_data,
        test_data       = test_data,
        device          = device,
    )

elif model_type == "fl_tgn":
    # Federated TGN — 4 independent clients, one checkpoint each.
    # Data is partitioned by source node ID (same split as training).
    # Global MRR = weighted average across clients by edge count.
    model = FederatedTGN(
        checkpoint_paths = checkpoints,
        num_nodes        = num_nodes,
        msg_dim          = msg_dim,
        train_data       = train_data,
        val_data         = val_data,
        test_data        = test_data,
        device           = device,
    )

elif model_type == "fedlink":
    # FedLink — static GraphSAGE, no temporal reasoning.
    # warmup() is a no-op; embeddings are recomputed fresh each eval call.
    model = FedLink(
        checkpoint_paths = checkpoints,
        num_users        = num_users,
        num_pages        = num_pages,
        train_data       = train_data,
        val_data         = val_data,
        test_data        = test_data,
        min_dst_idx      = min_dst_idx,
        device           = device,
    )

else:
    raise ValueError(f"Unknown model type: '{model_type}'. Choose from: tgn, fl_tgn, fedlink")

model.load_checkpoint()
print(f"Loaded: {model_name}")

## 5. Run Evaluation

In [ ]:
evaluator = Evaluator(
    dataset       = dataset,
    train_data    = train_data,
    val_data      = val_data,
    test_data     = test_data,
    first_dst_id  = min_dst_idx,
    last_dst_id   = max_dst_idx,
    dataset_name  = dataset_name,
    neg_cache_dir = neg_cache_dir,
    seed          = seed,
)

results = evaluator.run(model, model_name=model_name)

## 6. Results

In [ ]:
print(json.dumps(results, indent=2))